In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install sionna mitsuba

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.1/562.1 kB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 140.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 136.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 155.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.7/271.7 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 94.0 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Foun

Blender notes (for mitsuba scenes):
*    Use blender 3.6 + latest blosm & mitsuba
*    Use OpenStreetMap (OSM)
*    scene must be a triangle mesh
*   scene objects must have an itu material
*    mitsuba treats y-axis like the z-axis:
*    when finished, flip entire scene +90 x-axis
*    edit -> mesh -> bisect and knife project for cleaning map
*    must add blender plane for floor manually

In [ ]:
import os
import glob
import subprocess
import numpy as np
import mitsuba as mi
import torch
import sionna.rt
import matplotlib.pyplot as plt
import scipy
import pandas as pd

from sionna.rt import load_scene, Camera, Transmitter, Receiver, PlanarArray,\
    PathSolver, RadioMapSolver, load_mesh, watt_to_dbm, transform_mesh,\
    cpx_abs_square

class SyntheticMapGenerator():

    FLOOR_DB = -140.0
    MIN_DB = -145.0
    MAX_DB = -40.0     # often -35 to -50 depending on Tx power & distance

    def __init__(
        self,
        scene=None,
        num_samples=None,
        num_emitters=None,
        rm_arr_PRM=None,
        upper_bound=None,
        lower_bound=None,
        path_gain_db=None,
        rm_arr_np=None,
        verbose=None,       # toggle terminal messages
    ):
        self.verbose = True

    # load scene, scene bounds, and transmitter/receiver configuration for all transmitters/receivers
    def load_scene(self):
        filepath = '/content/drive/MyDrive/mitsuba_scenes/ucf_simplified/scene.xml'
        scene = load_scene(filepath) # scene with objects but flat
        scene.bandwidth=100e6
        self.scene = scene

        if not scene.objects:
            self.lower_bound = [-200,-200,1]
            self.upper_bound = [200,200,1]
        else:
            bbox = scene.mi_scene.bbox()

        # Get bounds (minimum X, Y, and Z coordinates)
        lower_bound = np.array(bbox.min)
        upper_bound = np.array(bbox.max)

        self.lower_bound = [round(lower_bound[0]),round(lower_bound[1]),1]
        self.upper_bound = [round(upper_bound[0]),round(upper_bound[1]),5]

        # print(f"Lower Bounds (min X, Y, Z): {lower_bound}")
        # print(f"Upper Bounds (max X, Y, Z): {upper_bound}")

        # Configure antenna arrays for all transmitters and receivers
        scene.tx_array = PlanarArray(num_rows=1,
                                    num_cols=1,
                                    pattern="iso", # tr38901, dipole, iso
                                    polarization="V"
                                    )
        scene.rx_array = PlanarArray(num_rows=1,
                                    num_cols=1,
                                    pattern="iso",
                                    polarization="V"
                                    )

    # finds transmitter positions that don't collide with scene objects using Mitsuba engine
    def find_pos(self, i, scene):
        lb = torch.tensor(self.lower_bound, dtype=torch.float32)
        ub = torch.tensor(self.upper_bound, dtype=torch.float32)

        mi_scene = self.scene.mi_scene
        valid_positions = []

        # finds random coords, creates rays pointing straight up and down at the coords and checks for collisions.
        # If ray doesn't intersect, add an emitter.
        # if ray intersects with mesh/object, try another random point.
        # Iterates until enough valid positions have been found.
        while len(valid_positions) < i:
            random_pos = lb + torch.rand((i, 3)) * (ub - lb) # random coords
            origins = mi.Point3f(random_pos[:, 0].numpy(),
                                random_pos[:, 1].numpy(),
                                random_pos[:, 2].numpy())
            dirs = mi.Vector3f(0.0, 0.0, 1.0)
            rays = mi.Ray3f(o=origins, d=dirs)
            si = mi_scene.ray_intersect(rays) # carries out ray collision
            hit = torch.tensor(np.array(si.is_valid())) # extract hit data via. np
            normal_z = torch.tensor(np.array(si.n.z))
            # collision-handling logic
            is_inside = hit & (normal_z > 0)
            is_valid = ~is_inside
            valid_batch = random_pos[is_valid]
            # add valid positions
            for pos in valid_batch:
                if len(valid_positions) < i:
                    valid_positions.append(pos)
                else:
                    break

        return torch.stack(valid_positions)

    # generate a num_samples-sized array of radio maps
    def generate_radio_maps(self):
        scene = self.scene
        self.num_samples = 100
        num_emitters = [np.random.randint(1, 1+1) for _ in range(self.num_samples)]

        self.rm_arr_np = []
        self.tx_meta = []
        self.rm_arr_PRM = [RadioMapSolver() for _ in range(self.num_samples)]
        for i in range(self.num_samples):
            for tx_name in list(scene.transmitters.keys()):
                    scene.remove(tx_name)

            # pos = np.random.uniform(lower_bounds, upper_bounds, (num_emitters[i],3))
            pos = self.find_pos(num_emitters[i], scene)
            pow = 50 #dbm

            for j in range(num_emitters[i]):

                current_orientation = [np.pi/6, 0, 0] if j == 0 else [-np.pi/2, 0, 0]

                tx = Transmitter(
                    name=f'tx{j}',
                    position=pos[j, :],
                    orientation=current_orientation,
                    power_dbm=pow
                )

                scene.add(tx)

            self.rm_arr_PRM[i] = RadioMapSolver()(
                            scene,
                            # measurement_surface=surface,   # for mesh radio map
                            specular_reflection=True,
                            diffuse_reflection=True,
                            refraction=True,
                            diffraction=True,
                            max_depth=10,           # Maximum number of ray scene interactions
                            samples_per_tx=(10 ** 7), # If you increase: less noise, but more memory required
                            cell_size=(1, 1),      # Resolution of the radio map
                            # center=[0, 0, 0],      # Center of the radio map
                            # orientation=[0, 0, 0] # Orientation of the radio map, e.g., could be also vertical
                        )

            # Converts radio map to np format and fills empty cells with db floor
            # NOTE: currently filled np radio map is NOT stored in rm_arr_PRM.
            for tx_idx in range(self.rm_arr_PRM[i].path_gain.shape[0]):
                path_gain_linear = self.rm_arr_PRM[i].path_gain[tx_idx]
                path_gain_np = np.array(path_gain_linear)

                print(f"Iteration: {i}")
                # print("Linear min/max:", path_gain_np.min(), path_gain_np.max())

                # Replace zeros with dB floor or set minimum (prevents divide w/ 0)
                pg_safe = np.maximum(path_gain_np, 10 ** (self.FLOOR_DB / 10.0))   # e.g. 10^(-140/10) = 1e-14

                # Convert to dB
                self.path_gain_db = 10.0 * np.log10(pg_safe)
                self.path_gain_db = np.clip(self.path_gain_db, self.MIN_DB, self.MAX_DB)

                print(f"Tx {tx_idx} — dB min/max: {self.path_gain_db.min():.2f} / {self.path_gain_db.max():.2f} dB")

                self.rm_arr_np.append(self.path_gain_db)

                # Labels
                tx_name = list(scene.transmitters.keys())[tx_idx]
                tx_obj = scene.transmitters[tx_name]
                pos_np = np.array(tx_obj.position)

                self.tx_meta.append({
                    'tx_x': float(pos_np[0][tx_idx]),
                    'tx_y': float(pos_np[1][tx_idx]),
                    'tx_z': float(pos_np[2][tx_idx]),
                    'power_dbm': float(np.array(tx_obj.power_dbm).item()),
                    'frequency_hz': float(np.array(scene.frequency).item())
                })
                # self.visualize_rm_np(tx_idx)


    # visualizes np radio map with matplotlib
    # NOTE: radio maps with n > 1 emitters are split into n radio maps, representing each emitter. Need to find a way to combine them.
    def visualize_rm_np(self, tx_idx):
        fig, ax = plt.subplots(figsize=(8, 8))

        im = ax.imshow(
            self.path_gain_db,
            cmap='jet',                     # many cool colors
            vmin=self.MIN_DB,
            vmax=self.MAX_DB,
            origin='lower',
            # interpolation='nearest'       # or 'bilinear'/'antialiased' if you want smoothing
        )

        ax.set_title(f"Tx {tx_idx} – Path gain [dB]")
        plt.colorbar(im, ax=ax, label="Path gain [dB]")
        plt.tight_layout()
        plt.show()
        plt.close(fig)

    # Visualizes radio map for PlanarRadioMap object
    def visualize_PRM(self):
        for i in range(self.num_samples):
            self.rm_arr_PRM[i].show(metric="path_gain", show_tx=False) # SINR, RSS, path_gain
            self.rm_arr_PRM[i].show_association(metric="path_gain")

    # Renders radio map
    def render_rm(self):
        cam = Camera(
                        position=[0,0,1500],
                        orientation=np.array([0,np.pi/2,-np.pi/2])
                        )
        self.scene.render(
            camera=cam,
            num_samples=1024,
            radio_map=self.rm_arr_PRM[-1],
            rm_metric="path_gain",
            rm_vmin=-120,
            rm_vmax=90
        )

    # Saves radio maps in parquet format
    def save_rm_parquet(self):
        parquet_dir = "samples_parquet"
        os.makedirs(parquet_dir, exist_ok=True)

        drive_dir = "/content/samples_parquet"
        existing_files = glob.glob(f"{drive_dir}/sample_*.parquet")
        if existing_files:
            # Extract numbers from filenames and find the max
            indices = [int(f.split('_')[-1].split('.')[0]) for f in existing_files]
            start_idx = max(indices) + 1
        else:
            start_idx = 0 # change to 0 for fresh dataset

        for i in range(self.num_samples):
            rm_np = self.rm_arr_np[i]
            meta = self.tx_meta[i]

            parquet_filename = f"sample_{(start_idx+i):04d}.parquet"
            parquet_path = os.path.join(parquet_dir, parquet_filename)

            data = {
                'sample_id': [start_idx + i],
                'tx_x': [meta['tx_x']],
                'tx_y': [meta['tx_y']],
                'tx_z': [meta['tx_z']],
                'power_dbm': [meta['power_dbm']],
                'frequency_hz': [meta['frequency_hz']],
                'map_width': [900],
                'map_height': [900],
                'map': [rm_np.astype('float32').flatten().tolist()] # Flatten the 2D array into a 1D list and store in a single column
            }
            df = pd.DataFrame(data)

            df.to_parquet(
                parquet_path,
                engine='pyarrow',
                compression='snappy', # snappy for faster writes, can use zstd
                )

        if self.verbose:
            print(f"Saved: {self.num_samples} sample(s) in parquet format")

    # Saves radio maps in jpg format
    def save_rm_png(self):
        png_dir = "samples_png"
        os.makedirs(png_dir, exist_ok=True)

        drive_dir = "/content/samples_png"
        existing_files = glob.glob(f"{drive_dir}/sample_*.png")
        if existing_files:
            # Extract numbers from filenames and find the max
            indices = [int(f.split('_')[-1].split('.')[0]) for f in existing_files]
            start_idx = max(indices) + 1
        else:
            start_idx = 0 # change to 0 for fresh dataset

        for i in range(self.num_samples):
            rm_np = self.rm_arr_np[i]
            png_filename = f"sample_{(start_idx+i):04d}.png"
            png_path = os.path.join(png_dir, png_filename)

            plt.imsave(
                png_path,
                rm_np,
                cmap='jet',
                vmin=self.MIN_DB,
                vmax=self.MAX_DB
            )

        if self.verbose:
            print(f"Saved: {self.num_samples} sample(s) in PNG format")

    def sync_to_drive(self, local_path, drive_path):
        print(f"Syncing {local_path} to {drive_path}...")
        # Using -u (update) flag to only copy new or changed files
        subprocess.run(["cp", "-ru", local_path, drive_path])
        print("Sync complete.")

    # # Saves radio maps in csv format
    # def save_rm_csv(self):
    #     csv_dir = "samples_csv"
    #     os.makedirs(csv_dir, exist_ok=True)

    #     for i in range(self.num_samples):
    #         rm_np = self.rm_arr_np[i]

    #         # saving CSV
    #         csv_filename = f"sample_{(i):04d}.csv"
    #         csv_path = os.path.join(csv_dir, csv_filename)

    #         np.savetxt(
    #             csv_path,
    #             rm_np,
    #             delimiter=',',
    #             fmt='%.8f',
    #             header="Path gain [dB] per cell",
    #             comments=''
    #         )

    #     if self.verbose:
    #             print(f"Saved: {self.num_samples} sample(s) in csv format")

In [ ]:
def main():
    generator = SyntheticMapGenerator()
    generator.load_scene()
    radio_maps = generator.generate_radio_maps()
    # generator.render_rm()
    generator.save_rm_png()
    generator.save_rm_parquet()
    generator.sync_to_drive("/content/samples_png/", "/content/drive/MyDrive/labeled_png_v2/training")
    generator.sync_to_drive("/content/samples_parquet/", "/content/drive/MyDrive/labels_parquet_v2/training")

if __name__ == "__main__":
    main()


Iteration: 0
Tx 0 — dB min/max: -140.00 / -53.82 dB
Iteration: 1
Tx 0 — dB min/max: -140.00 / -40.00 dB
Iteration: 2
Tx 0 — dB min/max: -140.00 / -51.65 dB
Iteration: 3
Tx 0 — dB min/max: -140.00 / -40.00 dB
Iteration: 4
Tx 0 — dB min/max: -140.00 / -51.02 dB
Iteration: 5
Tx 0 — dB min/max: -140.00 / -49.53 dB
Iteration: 6
Tx 0 — dB min/max: -140.00 / -50.61 dB
Iteration: 7
Tx 0 — dB min/max: -140.00 / -40.00 dB
Iteration: 8
Tx 0 — dB min/max: -140.00 / -48.43 dB
Iteration: 9
Tx 0 — dB min/max: -140.00 / -54.05 dB
Iteration: 10
Tx 0 — dB min/max: -140.00 / -54.19 dB
Iteration: 11
Tx 0 — dB min/max: -140.00 / -53.55 dB
Iteration: 12
Tx 0 — dB min/max: -140.00 / -40.00 dB
Iteration: 13
Tx 0 — dB min/max: -140.00 / -40.33 dB
Iteration: 14
Tx 0 — dB min/max: -140.00 / -40.00 dB
Iteration: 15
Tx 0 — dB min/max: -140.00 / -40.00 dB
Iteration: 16
Tx 0 — dB min/max: -140.00 / -50.13 dB
Iteration: 17
Tx 0 — dB min/max: -140.00 / -43.13 dB
Iteration: 18
Tx 0 — dB min/max: -140.00 / -40.00 dB
Ite